In [28]:
import numpy as np
import pandas as pd

trans_df = pd.read_csv(".data/medium.csv")
print("SIZE:", trans_df.size)
trans_df.head(5)

SIZE: 5499989


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [29]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/01 01:59]


In [30]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Series([], Name: Bank, dtype: int64)

In [31]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 188831


In [32]:
# Analyze accounts.
accounts_df = pd.read_csv(".data/LI-Small_accounts.csv")
print("SIZE:", accounts_df.shape[0])

SIZE: 712688


In [33]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 188831


In [34]:
ranged_trans_usd_sept_df = trans_usd_sept_1st_df\
    .groupby(["From Bank", "Account"])\
    .filter(lambda x: x.groupby(["To Bank", "Account.1"]).size().size > 5)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 4877


In [47]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']].sort_values(by=["From Bank", "Account"])
low_profile_transactions

,From Bank,Account,To Bank,Account.1,Amount Paid
188,1,800056ED0,1,800056ED0,21.08
33984,1,800056ED0,13243,805691140,26.58
197,1,80005A0E0,1,80005A0E0,15.33
449425,1,80005CA70,2368,805182900,0.32
301,1,8000F8F10,1,8000F8F10,23.37
...,...,...,...,...,...
162619,375009,81B719190,139951,81B719050,0.01
163513,375376,81B96BCA0,240478,81B94E7C0,0.02
164000,375513,81BA521F0,267983,81BA064C0,0.01
164724,376275,81BE1FC10,245810,81BE200B0,0.01


In [50]:
result_query1_client0 = pd.read_csv(".output/client0/query1.csv")
result_query1_client0 = result_query1_client0.sort_values(by=["from_bank", "from_account"])
result_query1_client0

,from_bank,from_account,to_bank,to_account,total_amount
71,1,800056ED0,1,800056ED0,21.08
10892,1,800056ED0,13243,805691140,26.58
85,1,80005A0E0,1,80005A0E0,15.33
54453,1,80005CA70,2368,805182900,0.32
107,1,8000F8F10,1,8000F8F10,23.37
...,...,...,...,...,...
53047,375009,81B719190,139951,81B719050,0.01
53345,375376,81B96BCA0,240478,81B94E7C0,0.02
53512,375513,81BA521F0,267983,81BA064C0,0.01
53728,376275,81BE1FC10,245810,81BE200B0,0.01


In [51]:
result_query1_client1 = pd.read_csv(".output/client1/query1.csv")
result_query1_client1 = result_query1_client1.sort_values(by=["from_bank", "from_account"])
result_query1_client1

,from_bank,from_account,to_bank,to_account,total_amount
91,1,800056ED0,1,800056ED0,21.08
10892,1,800056ED0,13243,805691140,26.58
67,1,80005A0E0,1,80005A0E0,15.33
54453,1,80005CA70,2368,805182900,0.32
112,1,8000F8F10,1,8000F8F10,23.37
...,...,...,...,...,...
53042,375009,81B719190,139951,81B719050,0.01
53323,375376,81B96BCA0,240478,81B94E7C0,0.02
53491,375513,81BA521F0,267983,81BA064C0,0.01
53720,376275,81BE1FC10,245810,81BE200B0,0.01


In [52]:
result_query2_client0 = pd.read_csv(".output/client0/query2.csv")
result_query2_client0 = result_query2_client0.sort_values(by="from_bank")
result_query2_client0

,from_bank,from_account,bank_name,amount_paid
1024,0,8060ADFF0,Italy Bank #18,5.457247e+05
1006,1,805611F00,First Bank of Portland,2.095891e+08
3868,3,80514DE70,Germany Bank #106,2.534500e+03
1068,7,80009A720,UK Bank #19,1.070160e+03
1689,8,80D5DD3E0,Canada Bank #0,3.018763e+04
...,...,...,...,...
1594,376850,81C1676B0,National Bank of Bridgeport,8.271400e+02
2729,376852,81C16A970,Dawn Federal Bank,1.215283e+04
586,376889,81C18B170,Desert Bank,1.724868e+04
3607,376932,81C1C9600,National Bank of New York,3.708250e+03


In [53]:
result_query2_client1 = pd.read_csv(".output/client1/query2.csv")
result_query2_client1 = result_query2_client1.sort_values(by="from_bank")
result_query2_client1

,from_bank,from_account,bank_name,amount_paid
1231,0,8060ADFF0,Italy Bank #18,5.457247e+05
6978,1,805611F00,First Bank of Portland,2.095891e+08
194,3,80514DE70,Germany Bank #106,2.534500e+03
5850,7,80009A720,UK Bank #19,1.070160e+03
3198,8,80D5DD3E0,Canada Bank #0,3.018763e+04
...,...,...,...,...
1147,376850,81C1676B0,National Bank of Bridgeport,8.271400e+02
3749,376852,81C16A970,Dawn Federal Bank,1.215283e+04
6941,376889,81C18B170,Desert Bank,1.724868e+04
6300,376932,81C1C9600,National Bank of New York,3.708250e+03


In [54]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
their_result = max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]].drop_duplicates()
their_result

,From Bank,Account,Bank Name,Amount Paid
0,0,8060ADFF0,Italy Bank #18,5.457247e+05
1236,1,805611F00,First Bank of Portland,2.095891e+08
2418,3,80514DE70,Germany Bank #106,2.534500e+03
3606,7,80009A720,UK Bank #19,1.070160e+03
4038,8,80D5DD3E0,Canada Bank #0,3.018763e+04
...,...,...,...,...
333876,376850,81C1676B0,National Bank of Bridgeport,8.271400e+02
333877,376852,81C16A970,Dawn Federal Bank,1.215283e+04
333878,376889,81C18B170,Desert Bank,1.724868e+04
333879,376932,81C1C9600,National Bank of New York,3.708250e+03


In [24]:
their_result.loc[their_result["From Bank"] == 31233]

,From Bank,Account,Bank Name,Amount Paid
57465,31233,80081AB80,National Bank of Cleveland,17599.0


In [39]:
for (bank_us, bank_them) in zip(results_1_df["from_bank"], their_result["From Bank"]):
    if bank_us != bank_them:
        print(f"Bank mismatch: {bank_us} != {bank_them}")

In [40]:
for (bank_us, bank_them) in zip(results_2_df["from_bank"], their_result["From Bank"]):
    if bank_us != bank_them:
        print(f"Bank mismatch: {bank_us} != {bank_them}")

In [ ]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]]

,From Bank,Account,Payment Format,Amount Paid
0,28932,806AC8450,ACH,22.95
1,140110,815D2D730,ACH,3126.23
4,10920,805364DE0,Cheque,28.19
6,70,10042B660,Cash,41.91
8,3304,8005E7A40,Cheque,118.48
...,...,...,...,...
83905,212518,80CF6A0F0,ACH,9746.50
83907,12,800057A30,ACH,58.45
83908,215187,805BB3960,ACH,6041.27
83909,11046,800BA56C0,Cheque,1803.61


In [ ]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
account_pairs_df = accounts_df.merge(accounts_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts

,Bank,Account
191,1217,800100AC0
306,22031,8023F6180
191,20,80011D080
306,42015,813703260
307,115966,80FD85F00


In [ ]:
#Conversion dates of period [2022-09-01, 2022-09-05] with base USD
#Bitcoin rates taken from investing.com
#Rest of currencies from api.frankfurter.dev
conversion_rates_records = np.rec.array([
           ('2022/09/01', 1.4644, 5.1805, 1.314 , 0.97999, 6.9   , 1.0002, 0.86272, 3.3535, 79.543, 139.34, 20.189, 60.367, 3.75, 1.,  19793.1),
           ('2022/09/02', 1.4691, 5.2035, 1.3141, 0.98175, 6.9035, 1.0011, 0.86468, 3.3755, 79.719, 140.11, 20.085, 60.427, 3.75, 1., 199999. ),
           ('2022/09/03', 1.4691, 5.2056, 1.3138, 0.98207, 6.9046, 1.0013, 0.86478, 3.3791, 79.75 , 140.17, 20.081, 60.471, 3.75, 1.,  19831.4),
           ('2022/09/04', 1.4695, 5.2082, 1.3139, 0.98219, 6.9047, 1.0013, 0.8649 , 3.3815, 79.754, 140.22, 20.084, 60.461, 3.75, 1.,  19952.7),
           ('2022/09/05', 1.4722, 5.1786, 1.3142, 0.98273, 6.9216, 1.0068, 0.86813, 3.4006, 79.816, 140.49, 20.018, 60.737, 3.75, 1.,  20126.1)],
          dtype=[ ('Date', 'O'), ('Australian Dollar', '<f8'), ('Brazil Real', '<f8'), ('Canadian Dollar', '<f8'), ('Swiss Franc', '<f8'), ('Yuan', '<f8'), ('Euro', '<f8'), ('UK Pound', '<f8'), ('Shekel', '<f8'), ('Rupee', '<f8'), ('Yen', '<f8'), ('Mexican Peso', '<f8'), ('Ruble', '<f8'), ('Saudi Riyal', '<f8'), ('US Dollar', '<f8'), ('Bitcoin', '<f8')])
conversion_rates_df = pd.DataFrame.from_records(conversion_rates_records)
conversion_rates_df = conversion_rates_df.set_index("Date")

In [ ]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df.apply(lambda row: row['Amount Paid'] / conversion_rates_df[row['Payment Currency']][row["Timestamp"].split(" ")[0]], axis=1)
trans_sept_1st_wire_or_ach_filtered = trans_sept_1st_wire_or_ach_converted_df[trans_sept_1st_wire_or_ach_converted_df['Amount'] < 1.0]
print("SIZE:", trans_sept_1st_wire_or_ach_filtered.shape[0])

SIZE: 689
